In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install streamlit faiss-cpu langchain langchain-community langchain-core langchain-huggingface pypdf sentence-transformers transformers==4.52.4 torch accelerate -q
!npm install localtunnel -q
print("======= Dependencies Installed =======")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 74.5 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 96.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 79.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 94.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 74.8 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency con

In [11]:
%%writefile app.py
import streamlit as st
import torch
import re
import tempfile
import os
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_classic.output_parsers import ResponseSchema
from langchain_core.prompts import PromptTemplate

# 1. Page Configuration
st.set_page_config(page_title="AI Interview Simulator", page_icon="🤖", layout="wide")
st.title("🤖 AI Technical Interview Simulator")

# 2. Cache Heavy Models 
@st.cache_resource
def load_models():
    model_name = "mistralai/Mistral-Nemo-Instruct-2407"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, 
        torch_dtype=torch.float16, 
        device_map="auto"
    )
    embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return tokenizer, model, embedding

tokenizer, model, embedding = load_models()

def generate_text(prompt, max_length=1500, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs][0]

def extract_json_block(text):
    # Try to find JSON inside triple backticks first
    pattern = r'```(?:json)?\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)
    if matches:
        try:
            return json.loads(matches[-1])
        except Exception:
            pass
            
    # Fallback: try parsing the whole text or finding curly braces
    try:
        start = text.index('{')
        end = text.rindex('}') + 1
        return json.loads(text[start:end])
    except Exception:
        return {
            "score": "N/A",
            "missing_points": text,
            "improvement_suggestions": "Could not extract structured JSON cleanly, review raw output."
        }

# 3. Setup LangChain Schemas & Prompts
question_schema = ResponseSchema(name="question", description="The interview question asked.") 
answer_schema = ResponseSchema(name="answer", description="The candidate's provided answer.")
score_schema = ResponseSchema(name="score", description="A score out of 10 for the answer.")
missing_schema = ResponseSchema(name="missing_points", description="Key technical points the candidate failed to mention.")
improvement_schema = ResponseSchema(name="improvement_suggestions", description="Actionable feedback to improve the answer.")

question_generation_template = """
You are an expert technical recruiter. Review the following background extracted from a candidate's CV:
{context}

Previously asked questions (DO NOT ASK THESE AGAIN):
{asked_questions}

Based on their specific experiences, projects, and skills, generate ONE new, challenging but fair technical interview question tailored exactly to their background. 
Do not include any conversational filler, greetings, or explanations. Output ONLY the question.
"""

evaluation_template = """
You are an expert technical interviewer evaluating a candidate.
Here is the relevant background from the candidate's CV: {context}

You previously asked the candidate this question: "{interview_question}"
The candidate provided this answer: "{candidate_answer}"

Evaluate the candidate's answer based on technical accuracy, completeness, and communication.
Respond ONLY in valid JSON format containing these exact keys: "score", "missing_points", "improvement_suggestions".
"""

# 4. Session State Management
if "asked_questions" not in st.session_state:
    st.session_state.asked_questions = []
if "current_question" not in st.session_state:
    st.session_state.current_question = None
if "context_text" not in st.session_state:
    st.session_state.context_text = ""
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

# 5. Sidebar UI for Dynamic CV Upload
with st.sidebar:
    st.header("📄 Upload Candidate CV")
    uploaded_file = st.file_uploader("Upload a PDF file to begin", type="pdf")
    
    if uploaded_file is not None and st.session_state.context_text == "":
        with st.spinner("Processing CV and building Vector Database..."):
            with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp_file:
                tmp_file.write(uploaded_file.getvalue())
                tmp_path = tmp_file.name
            
            cv_loader = PyPDFLoader(tmp_path)
            cv_docs = cv_loader.load()
            full_cv_text = "\n\n".join([doc.page_content for doc in cv_docs])
            
            text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=100)
            cv_chunks = text_splitter.split_documents(cv_docs)
            vectordb = FAISS.from_documents(cv_chunks, embedding)
            
            if len(full_cv_text) < 2000:
                st.session_state.context_text = full_cv_text
            else:
                retriever = vectordb.as_retriever(search_kwargs={"k": 4})
                candidate_context = retriever.invoke("technical skills, work experience, main projects, tools, education, and achievements")
                st.session_state.context_text = "\n".join([doc.page_content for doc in candidate_context])
            
            os.remove(tmp_path)
            st.success("CV Loaded Successfully! You can now start the interview.")

# 6. Main Chat Interface
if st.session_state.context_text != "":
    # Render previous chat history
    for msg in st.session_state.chat_history:
        with st.chat_message(msg["role"]):
            st.markdown(msg["content"])
            if "evaluation" in msg:
                with st.expander("📊 View Detailed Evaluation"):
                    eval_data = msg["evaluation"]
                    if isinstance(eval_data, str):
                        eval_data = extract_json_block(eval_data)
                    
                    st.markdown(f"**Score:** {eval_data.get('score', 'N/A')}/10")
                    st.markdown(f"**Missing Points:** {eval_data.get('missing_points', '')}")
                    st.markdown(f"**Suggestions:** {eval_data.get('improvement_suggestions', '')}")

    # Generate Question Logic
    if st.session_state.current_question is None:
        if st.button("Generate Next Interview Question"):
            with st.spinner("Analyzing CV and generating a tailored question..."):
                q_prompt = PromptTemplate(
                    template=question_generation_template,
                    input_variables=["context", "asked_questions"]
                ).format(
                    context=st.session_state.context_text, 
                    asked_questions="\n".join(st.session_state.asked_questions) if st.session_state.asked_questions else "None"
                )
                new_q = generate_text(q_prompt, max_length=5000).strip()
                
                st.session_state.current_question = new_q
                st.session_state.asked_questions.append(new_q)
                st.session_state.chat_history.append({"role": "assistant", "content": new_q})
                st.rerun()

    # Chat Input Logic
    if st.session_state.current_question is not None:
        answer = st.chat_input("Type your answer here...")
        if answer:
            st.session_state.chat_history.append({"role": "user", "content": answer})
            
            with st.spinner("Evaluating your answer..."):
                eval_prompt = PromptTemplate(
                    template=evaluation_template,
                    input_variables=["context", "interview_question", "candidate_answer"]
                ).format(
                    context=st.session_state.context_text,
                    interview_question=st.session_state.current_question,
                    candidate_answer=answer
                )
                raw_response = generate_text(eval_prompt, max_length=5000)
                parsed_json = extract_json_block(raw_response)
                
                # Attach the clean parsed evaluation dictionary to the message
                st.session_state.chat_history[-1]["evaluation"] = parsed_json
                
                st.session_state.current_question = None
                st.rerun()
else:
    st.info("👈 Please upload a Candidate CV in the sidebar to begin the simulation.")

Overwriting app.py


In [ ]:
!pip install pyngrok -q

from pyngrok import ngrok
import os
import time

# 1. Clear out any old processes so they don't conflict
os.system("pkill -f streamlit")
os.system("pkill -f ngrok")
time.sleep(2)

# 2. Set your ngrok token (paste your token between the quotes below)
ngrok.set_auth_token("PUT YOUR NGROK TOKEN HERE")

# 3. Start Streamlit in the background
os.system("streamlit run app.py &")
time.sleep(3)

# 4. Create and print the clean ngrok tunnel link
try:
    public_url = ngrok.connect(8501).public_url
    print(f"======> 🚀 Your app is live at: {public_url}")
except Exception as e:
    print(f"Failed to start ngrok: {e}")

  Stopping...




2026-07-24 22:59:05.274 Uvicorn server started on :::8501



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://34.44.240.181:8501

======> 🚀 Your app is live at: https://faster-moonlike-fretful.ngrok-free.dev


/kaggle/working/app.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
Loading checkpoint shards: 100%|██████████| 5/5 [01:51<00:00, 22.26s/it]
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


────────────────────────── Traceback (most recent call last) ───────────────────────────
  /usr/local/lib/python3.12/dist-packages/streamlit/runtime/scriptrunner/exec_code.py:  
  129 in exec_func_with_error_handling                                                  
                                                                                        
  /usr/local/lib/python3.12/dist-packages/streamlit/runtime/scriptrunner/script_runner  
  .py:807 in code_to_exec                                                               
                                                                                        
  /kaggle/working/app.py:186 in <module>                                                
                                                                                        
    183 │   │   │   │   │   interview_question=st.session_state.current_question,       
    184 │   │   │   │   │   candidate_answer=answer                                     
    185 │   │   │   │

...

────────────────────────── Traceback (most recent call last) ───────────────────────────
  /usr/local/lib/python3.12/dist-packages/streamlit/runtime/scriptrunner/exec_code.py:  
  129 in exec_func_with_error_handling                                                  
                                                                                        
  /usr/local/lib/python3.12/dist-packages/streamlit/runtime/scriptrunner/script_runner  
  .py:807 in code_to_exec                                                               
                                                                                        
  /kaggle/working/app.py:164 in <module>                                                
                                                                                        
    161 │   │   │   │   │   context=st.session_state.context_text,                      
    162 │   │   │   │   │   asked_questions="\n".join(st.session_state.asked_questions  
    163 │   │   │   │

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


────────────────────────── Traceback (most recent call last) ───────────────────────────
  /usr/local/lib/python3.12/dist-packages/streamlit/runtime/scriptrunner/exec_code.py:  
  129 in exec_func_with_error_handling                                                  
                                                                                        
  /usr/local/lib/python3.12/dist-packages/streamlit/runtime/scriptrunner/script_runner  
  .py:807 in code_to_exec                                                               
                                                                                        
  /kaggle/working/app.py:164 in <module>                                                
                                                                                        
    161 │   │   │   │   │   context=st.session_state.context_text,                      
    162 │   │   │   │   │   asked_questions="\n".join(st.session_state.asked_questions  
    163 │   │   │   │

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


────────────────────────── Traceback (most recent call last) ───────────────────────────
  /usr/local/lib/python3.12/dist-packages/streamlit/runtime/scriptrunner/exec_code.py:  
  129 in exec_func_with_error_handling                                                  
                                                                                        
  /usr/local/lib/python3.12/dist-packages/streamlit/runtime/scriptrunner/script_runner  
  .py:807 in code_to_exec                                                               
                                                                                        
  /kaggle/working/app.py:164 in <module>                                                
                                                                                        
    161 │   │   │   │   │   context=st.session_state.context_text,                      
    162 │   │   │   │   │   asked_questions="\n".join(st.session_state.asked_questions  
    163 │   │   │   │